In [1]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator


In [2]:
# Pra-pengolahan citra data training dan testing
train_datagen = ImageDataGenerator(
    rescale=1./255,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True
)
training_set = train_datagen.flow_from_directory(
    'dataset/training_set',
    target_size=(64, 64),
    batch_size=32,
    class_mode='binary'
)

test_datagen = ImageDataGenerator(rescale=1./255)
test_set = test_datagen.flow_from_directory(
    'dataset/test_set',
    target_size=(64, 64),
    batch_size=32,
    class_mode='binary'
)


Found 8000 images belonging to 2 classes.


Found 2000 images belonging to 2 classes.


In [3]:
tf.random.set_seed(42)

# Bangun arsitektur model CNN
cnn = tf.keras.models.Sequential([
    tf.keras.layers.Conv2D(filters=32, kernel_size=3, activation='relu', input_shape=[64, 64, 3]),
    tf.keras.layers.MaxPool2D(pool_size=2, strides=2),
    tf.keras.layers.Conv2D(filters=32, kernel_size=3, activation='relu'),
    tf.keras.layers.MaxPool2D(pool_size=2, strides=2),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(units=128, activation='relu'),
    tf.keras.layers.Dense(units=1, activation='sigmoid')
])

cnn.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
cnn.summary()


C:\Users\azure\anaconda3\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 62, 62, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 31, 31, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 29, 29, 32)     │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 14, 14, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 6272)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       802,944 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 813,217 (3.10 MB)

 Trainable params: 813,217 (3.10 MB)

 Non-trainable params: 0 (0.00 B)

In [4]:
# Pelatihan model CNN
history = cnn.fit(
    x=training_set,
    validation_data=test_set,
    epochs=10,
    steps_per_epoch=100,
    validation_steps=30
)


Epoch 1/10


  1/100 ━━━━━━━━━━━━━━━━━━━━ 2:46 2s/step - accuracy: 0.4375 - loss: 0.7231

  3/100 ━━━━━━━━━━━━━━━━━━━━ 6s 71ms/step - accuracy: 0.5104 - loss: 0.8289

  4/100 ━━━━━━━━━━━━━━━━━━━━ 8s 88ms/step - accuracy: 0.4688 - loss: 0.8171

  5/100 ━━━━━━━━━━━━━━━━━━━━ 8s 93ms/step - accuracy: 0.4875 - loss: 0.7906

  6/100 ━━━━━━━━━━━━━━━━━━━━ 9s 97ms/step - accuracy: 0.4792 - loss: 0.7774

... (851 outputs left out)

 98/100 ━━━━━━━━━━━━━━━━━━━━ 0s 119ms/step - accuracy: 0.6869 - loss: 0.5836

 99/100 ━━━━━━━━━━━━━━━━━━━━ 0s 119ms/step - accuracy: 0.6881 - loss: 0.5829

100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 119ms/step - accuracy: 0.6881 - loss: 0.5827

100/100 ━━━━━━━━━━━━━━━━━━━━ 14s 140ms/step - accuracy: 0.6881 - loss: 0.5827 - val_accuracy: 0.7219 - val_loss: 0.5590


In [5]:
import numpy as np
from tensorflow.keras.preprocessing import image

# Prediksi citra tunggal 1
test_image_1 = image.load_img('dataset/single_prediction/cat_or_dog_1.jpg', target_size=(64, 64))
test_image_1 = image.img_to_array(test_image_1)
test_image_1 = np.expand_dims(test_image_1, axis=0) / 255.0
result_1 = cnn.predict(test_image_1)

pred_1 = 'dog' if result_1[0][0] > 0.5 else 'cat'
print(f'Probabilitas cat_or_dog_1.jpg: {result_1[0][0]:.4f}')
print('Hasil prediksi:', pred_1)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 184ms/step

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 214ms/step


Probabilitas cat_or_dog_1.jpg: 0.9072
Hasil prediksi: dog


In [6]:
# Prediksi citra tunggal 2
test_image_2 = image.load_img('dataset/single_prediction/cat_or_dog_2.jpg', target_size=(64, 64))
test_image_2 = image.img_to_array(test_image_2)
test_image_2 = np.expand_dims(test_image_2, axis=0) / 255.0
result_2 = cnn.predict(test_image_2)

pred_2 = 'dog' if result_2[0][0] > 0.5 else 'cat'
print(f'Probabilitas cat_or_dog_2.jpg: {result_2[0][0]:.4f}')
print('Hasil prediksi:', pred_2)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 87ms/step


Probabilitas cat_or_dog_2.jpg: 0.4301
Hasil prediksi: cat
